# 01 — Business Understanding
### Manufacturing Quality Analytics: Root Cause Analysis of Semiconductor Test Failures

**Author's note:** This notebook is intentionally code-light. Before touching data, the goal is
to be explicit about the business problem being solved, who the stakeholders are, and what
"success" looks like — so every later technical decision can be justified against these criteria.


## 1. The Business Problem

A semiconductor fabrication line runs each wafer/unit through an in-line test process that
records readings from **590 process sensors** (temperature, pressure, flow, electrical
signatures, etc.) per unit, alongside a final **Pass/Fail** test outcome.

Engineers currently have no fast way to know *which* of the 590 sensors are actually
associated with failures. When a batch of units fails, root-cause investigation is manual,
slow, and reactive — by the time a cause is found, more defective units may already be in
production.

```
Manufacturing Line
        │
        ▼
   590 Process Sensors per Unit
        │
        ▼
 Engineers overwhelmed by dimensionality
        │
        ▼
 Need: identify failure-driving sensors
        │
        ▼
 Outcome: reduce defective production, target monitoring/recalibration
```


## 2. Why This Matters

- **Cost of escapes:** every failed unit that reaches later stages (or the customer) is far
  more expensive to catch than one flagged in-line.
- **Cost of over-inspection:** blanket manual inspection of all 590 sensors after every run is
  not scalable — engineers need a *prioritized* list of a handful of sensors to watch.
- **Yield improvement:** even a modest, statistically defensible narrowing from 590 sensors to
  a validated shortlist of ~15 gives engineering teams a concrete, actionable place to start
  (recalibration, tighter control limits, maintenance scheduling).

## 3. Objectives

1. Build a reliable, reproducible data-cleaning pipeline for the raw sensor data.
2. Identify which sensors are most predictive of failure, using a model-based approach
   (Random Forest feature importance).
3. **Statistically validate** that those "important" sensors actually differ between pass and
   fail units — a model can overfit or pick up spurious importance, so validation matters.
4. Quantify how well those sensors, combined, can *detect* failures (classification
   performance), being honest about the limits imposed by a ~6.6% failure rate and only
   1,567 units.
5. Package findings into an interactive dashboard and a set of concrete engineering
   recommendations.

## 4. Success Criteria

| Criterion | Target |
|---|---|
| Data cleaning | Documented, reproducible drop/impute rules (no silent data loss) |
| Feature identification | Top sensors ranked by model importance |
| Statistical validation | Mann-Whitney U + multiple-comparison correction on flagged sensors |
| Model performance | Report ROC-AUC / average precision honestly (not accuracy, given class imbalance) |
| Communication | Non-technical business recommendations a process engineer could act on directly |

## 5. Scope & Limitations (stated up front)

- This dataset has **no sensor names/units** — only numeric IDs. Recommendations will be framed
  as "Sensor 59", "Sensor 130", etc., the way an analyst would communicate to engineers who
  *do* know what those IDs map to on the line.
- With only 104 failure examples, any model is working with a small positive class — this
  project treats the model as a **prioritization tool for engineers**, not a fully automated
  pass/fail decision system, and says so explicitly in the final recommendations.
